In [ ]:
import numpy as np

class IMMFilter:
    def __init__(self, initial_state, initial_covariance, transition_matrix, measurement_matrix, process_noise_covs, measurement_noise_cov, jump_sizes, transition_probs):
        self.num_models = len(jump_sizes) + 1  # +1 for no jump model
        self.models = []
        for i in range(self.num_models):
            # Define process noise for each model
            if i==0:
              Q = process_noise_covs[0]
            else:
              Q = process_noise_covs[1]
            kf = KalmanFilter(initial_state, initial_covariance, transition_matrix, measurement_matrix, Q, measurement_noise_cov)
            self.models.append(kf)

        self.jump_sizes = [1.0] + jump_sizes  # 1.0 for no jump
        self.transition_probs = transition_probs # transition_probs[i][j] is the probability from model i to model j
        self.model_probs = np.ones(self.num_models) / self.num_models  # Initial model probabilities

    def predict(self, measurement):

        # 1. Prediction Step for Each Model
        for i in range(self.num_models):
            self.models[i].predict()

        # 2. Update Model Probabilities
        likelihoods = np.zeros(self.num_models)
        for i in range(self.num_models):
            likelihoods[i] = self.models[i].likelihood(measurement)

        # Mixing probabilities
        mixed_model_probs = np.zeros(self.num_models)
        for j in range(self.num_models):
          for i in range(self.num_models):
            mixed_model_probs[j] += self.transition_probs[i][j] * self.model_probs[i]
        
        #Update model probabilities
        for i in range(self.num_models):
            self.model_probs[i] = likelihoods[i] * mixed_model_probs[i]
        self.model_probs /= np.sum(self.model_probs)  # Normalize

        # 3. Combined State Estimate
        combined_state = np.zeros_like(self.models[0].state)
        for i in range(self.num_models):
            combined_state += self.model_probs[i] * self.models[i].state

        return combined_state

class KalmanFilter:
    def __init__(self, initial_state, initial_covariance, transition_matrix, measurement_matrix, process_noise_cov, measurement_noise_cov):
        self.state = initial_state
        self.covariance = initial_covariance
        self.F = transition_matrix
        self.H = measurement_matrix
        self.Q = process_noise_cov
        self.R = measurement_noise_cov

    def predict(self):
        self.state = self.F @ self.state
        self.covariance = self.F @ self.covariance @ self.F.T + self.Q

    def update(self, measurement):
        y = measurement - self.H @ self.state
        S = self.H @ self.covariance @ self.H.T + self.R
        K = self.covariance @ self.H.T @ np.linalg.inv(S)
        self.state = self.state + K @ y
        self.covariance = (np.eye(len(self.state)) - K @ self.H) @ self.covariance

    def likelihood(self, measurement):
        y = measurement - self.H @ self.state
        S = self.H @ self.covariance @ self.H.T + self.R
        return np.exp(-0.5 * y.T @ np.linalg.inv(S) @ y) / np.sqrt(2 * np.pi * np.linalg.det(S))

# Example Usage
initial_state = np.array([0.0, 0.0])  # [P, r]
initial_covariance = np.eye(2) * 1 # 0.01
transition_matrix = np.array([[1.0, 0.0], [0.0, 1.0]])  # Placeholder, r model needed
measurement_matrix = np.array([[1.0, 0.0]])  # Observing P
process_noise_covs = [np.eye(2) * 0.0001, np.eye(2) * 0.01]  # Process noise for no jump and jump models
measurement_noise_cov = np.array([[0.01]])  # Measurement noise
jump_sizes = [2.0, 5.0]  # Example jump sizes
transition_probs = np.array([[0.95, 0.02, 0.03], [0.05, 0.9, 0.05], [0.05, 0.05, 0.9]]) # Example transition probabilities

imm_filter = IMMFilter(initial_state, initial_covariance, transition_matrix, measurement_matrix, process_noise_covs, measurement_noise_cov, jump_sizes, transition_probs)

# Example measurements (replace with your actual data)
measurements = [0.1, 0.2, 0.3, 0.8, 0.9, 1.0, 1.1] # Jump at measurement 4
measurements = [ 0.004,  0.002,  0.002,  0.003,  0.001, -0.001, -0.   , -0.003,
       -0.003,  0.002,  0.001, -0.   ,  0.003, -0.   ,  0.001,  0.002,
       -0.001, -0.   ,  0.001, -0.001,  0.   , -0.   ,  0.001,  0.   ,
        0.001,  0.003,  0.002, -0.   ,  0.001, -0.   ,  0.001,  0.002,
        0.   , -0.002, -0.001, -0.   , -0.001,  0.002, -0.002,  0.001,
        0.002,  0.003, -0.001, -0.001,  0.001,  0.002,  0.   , -0.   ,
        0.001]
for measurement in measurements:
    estimated_state = imm_filter.predict(np.array([measurement]))
    print("Estimated state:", estimated_state)